# Aula 6.12 — Exercício guiado: Perceptron e MLP com Backpropagation

**Experimento:** construir um Perceptron e, depois, uma rede neural de duas camadas para aprender a função XOR usando somente NumPy.

Neste notebook, vamos construir tudo em uma sessão nova: imports, dados, funções, parâmetros, treinamento e gráficos. Não é necessário executar nenhum código dos slides antes.

**Objetivos:**
- Treinar um Perceptron para aprender OR.
- Verificar experimentalmente por que um único Perceptron não resolve XOR.
- Construir uma MLP com arquitetura `2 → 4 → 1`.
- Implementar `forward`, Binary Cross-Entropy, `backward` e atualização dos parâmetros.
- Treinar a rede e conferir as quatro saídas do XOR.

> O exercício utiliza as quatro linhas da tabela XOR como conjunto completo de treinamento. Não fazemos divisão treino/teste porque a finalidade aqui é aprender e reproduzir integralmente essa função lógica minúscula, não estimar a generalização para uma população maior.

## Etapa 1 — Importar as bibliotecas

Usaremos apenas:
- **NumPy:** arrays, operações matriciais, funções matemáticas e números aleatórios.
- **Matplotlib:** gráficos de dispersão, curva de perda e região de decisão.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## Etapa 2 — Criar a tabela XOR

A função XOR retorna `1` quando as entradas são diferentes e `0` quando são iguais.

- `X` contém quatro amostras e duas características: `x1` e `x2`.
- `Y` contém os rótulos verdadeiros como uma coluna (`4 × 1`).
- Usamos `dtype=float` para permitir operações com pesos reais.

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

Y = np.array([
    [0],
    [1],
    [1],
    [0]
], dtype=float)

print("Formato de X:", X.shape)
print("Formato de Y:", Y.shape)
print("Entradas:\n", X)
print("Saídas XOR:\n", Y.ravel())

## Etapa 3 — Visualizar o XOR

Cada ponto representa uma combinação de entrada. A cor indica a classe.

Observe que os dois pontos da classe 1 ficam em posições diagonais opostas, assim como os dois pontos da classe 0. Essa disposição é a pista visual de que uma única reta não consegue separar as classes perfeitamente.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    X[:, 0], X[:, 1],
    c=Y.ravel(), cmap="coolwarm", s=300,
    edgecolors="black", linewidths=1.2
)
for (x1, x2), y in zip(X, Y.ravel()):
    plt.annotate(f"{int(y)}", (x1, x2), xytext=(0, 10),
                 textcoords="offset points", ha="center", fontsize=12)
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("XOR: classes em diagonais opostas")
plt.grid(alpha=0.25)
plt.show()

## Etapa 4 — Preparar os rótulos do problema OR

Antes de tentar a MLP, vamos implementar um Perceptron simples.

Para seguir a regra clássica de atualização do Perceptron, representaremos as classes por `-1` e `+1`:

- `0` lógico → `-1`;
- `1` lógico → `+1`.

A ordem das amostras continua sendo a mesma de `X`.

In [ ]:
Y_or = np.array([-1, 1, 1, 1])
print("Rótulos OR em -1/+1:", Y_or)

## Etapa 5 — Implementar o treinamento do Perceptron

O Perceptron começa com pesos e bias iguais a zero. Para cada amostra:

1. Calcula `z = x · pesos + bias`.
2. Prevê `+1` quando `z >= 0`; caso contrário, prevê `-1`.
3. Se errar, atualiza pesos e bias:

\[
\mathbf{w} \leftarrow \mathbf{w} + \eta y_i\mathbf{x}_i,
\qquad
b \leftarrow b + \eta y_i.
\]

`η` é a taxa de aprendizado; `y_i` é o rótulo verdadeiro; `x_i` é a amostra. O código percorre uma amostra por vez (atualização online).

In [ ]:
def treinar_perceptron(X, y, taxa_aprendizado=0.1, max_epocas=100):
    pesos = np.zeros(X.shape[1])
    bias = 0.0

    for epoca in range(max_epocas):
        erros = 0

        for xi, yi in zip(X, y):
            z = np.dot(xi, pesos) + bias
            previsao = 1 if z >= 0 else -1

            if previsao != yi:
                pesos += taxa_aprendizado * yi * xi
                bias += taxa_aprendizado * yi
                erros += 1

        if erros == 0:
            break

    return pesos, bias, epoca + 1

## Etapa 6 — Treinar e avaliar o Perceptron no OR

Agora chamamos a função que acabamos de definir. O problema OR é linearmente separável; portanto, o Perceptron consegue encontrar uma fronteira que classifique as quatro amostras.

In [ ]:
pesos_or, bias_or, epocas_or = treinar_perceptron(X, Y_or)
z_or = X @ pesos_or + bias_or
previsoes_or = np.where(z_or >= 0, 1, -1)

print("Pesos:", pesos_or)
print("Bias:", bias_or)
print("Épocas:", epocas_or)
print("Previsões:", previsoes_or)
print("Esperado: ", Y_or)
print("Acertos:", np.sum(previsoes_or == Y_or), "de", len(Y_or))

## Etapa 7 — Tentar aprender XOR com o mesmo Perceptron

Agora representamos XOR também com rótulos `-1/+1`. Mantemos o mesmo algoritmo e limitamos o treinamento a 100 épocas.

O Perceptron não encontra uma solução sem erros porque XOR não é linearmente separável. A execução é uma demonstração empírica dessa limitação; a justificativa geométrica e algébrica aparece na parte teórica da aula.

In [ ]:
Y_xor_pm = np.array([-1, 1, 1, -1])

pesos_xor, bias_xor, epocas_xor = treinar_perceptron(
    X, Y_xor_pm, taxa_aprendizado=0.1, max_epocas=100
)

z_xor = X @ pesos_xor + bias_xor
previsoes_xor = np.where(z_xor >= 0, 1, -1)

print("Épocas executadas:", epocas_xor)
print("Pesos:", pesos_xor)
print("Bias:", bias_xor)
print("Previsões:", previsoes_xor)
print("Esperado:  ", Y_xor_pm)
print("Acertos:", np.sum(previsoes_xor == Y_xor_pm), "de", len(Y_xor_pm))

## Etapa 8 — Definir a função Sigmoid

A MLP utilizará a função sigmoid como ativação dos neurônios:

\[
\sigma(z)=\frac{1}{1+e^{-z}}.
\]

Ela transforma qualquer número real em um valor entre 0 e 1. `np.clip` limita os valores extremos antes de calcular a exponencial, reduzindo o risco de overflow numérico.

In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

print("Sigmoid(-2), sigmoid(0), sigmoid(2):", sigmoid(np.array([-2.0, 0.0, 2.0])))

## Etapa 9 — Inicializar os parâmetros da MLP

Vamos utilizar a arquitetura:

**2 entradas → 4 neurônios ocultos → 1 saída.**

As dimensões são:
- `W1`: `2 × 4`; `b1`: `1 × 4`.
- `W2`: `4 × 1`; `b2`: `1 × 1`.

Os pesos começam aleatórios para quebrar a simetria entre neurônios. Os biases começam em zero. A semente `42` torna a inicialização reproduzível.

In [ ]:
rng = np.random.default_rng(42)

n_entradas = 2
n_ocultos = 4
n_saidas = 1

params = {
    "W1": rng.normal(0, 0.8, size=(n_entradas, n_ocultos)),
    "b1": np.zeros((1, n_ocultos)),
    "W2": rng.normal(0, 0.8, size=(n_ocultos, n_saidas)),
    "b2": np.zeros((1, n_saidas))
}

for nome, valor in params.items():
    print(nome, valor.shape)

## Etapa 10 — Implementar o Forward Propagation

O forward calcula a saída camada por camada:

\[
Z^{[1]}=XW^{[1]}+b^{[1]},\qquad A^{[1]}=\sigma(Z^{[1]})
\]
\[
Z^{[2]}=A^{[1]}W^{[2]}+b^{[2]},\qquad A^{[2]}=\sigma(Z^{[2]}).
\]

O dicionário `cache` guarda os valores intermediários necessários no Backpropagation. Para as quatro amostras, `A2` terá formato `4 × 1`.

In [ ]:
def forward(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = sigmoid(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = sigmoid(Z2)

    cache = {
        "X": X,
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache

A2_inicial, cache_inicial = forward(X, params)
print("Previsões iniciais A2:", A2_inicial.ravel())
print("Formato A1:", cache_inicial["A1"].shape)
print("Formato A2:", A2_inicial.shape)

## Etapa 11 — Implementar a Binary Cross-Entropy

A saída da sigmoid é comparada com o rótulo verdadeiro por meio da Binary Cross-Entropy:

\[
L=-\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(a_i)+(1-y_i)\log(1-a_i)\right].
\]

`m` é o número de amostras, `y` é o rótulo verdadeiro e `a` é a previsão. O `epsilon` evita calcular `log(0)`. A perda inicial deve ser aproximadamente `0.7285`, usando a semente `42`.

In [ ]:
def calcular_loss(Y, probabilidades):
    epsilon = 1e-12
    probabilidades = np.clip(probabilidades, epsilon, 1 - epsilon)

    loss = -np.mean(
        Y * np.log(probabilidades)
        + (1 - Y) * np.log(1 - probabilidades)
    )

    return float(loss)

loss_inicial = calcular_loss(Y, A2_inicial)
print("Perda inicial:", loss_inicial)

## Etapa 12 — Entender a Regra da Cadeia com um exemplo numérico

Considere uma única amostra, com saída prevista `A2 = 0.8`, rótulo verdadeiro `Y = 1`, ativação oculta `A1 = 0.6` e peso de conexão `W2 = 2`.

Para a combinação sigmoid + BCE, o gradiente em relação ao valor antes da sigmoid de saída é:

\[
dZ_2=A_2-Y=0.8-1=-0.2.
\]

Retornando para o neurônio oculto:

\[
dA_1=dZ_2W_2=-0.2\cdot2=-0.4
\]
\[
dZ_1=dA_1\,A_1(1-A_1)=-0.4\cdot0.6\cdot0.4=-0.096.
\]

Este exemplo ilustra uma amostra única. No código de treinamento em lote, dividiremos por `m` para calcular a perda média.

In [ ]:
A2_exemplo = 0.8
Y_exemplo = 1.0
A1_exemplo = 0.6
W2_exemplo = 2.0

dZ2_exemplo = A2_exemplo - Y_exemplo
dA1_exemplo = dZ2_exemplo * W2_exemplo
dZ1_exemplo = dA1_exemplo * A1_exemplo * (1 - A1_exemplo)

print("dZ2 =", dZ2_exemplo)
print("dA1 =", dA1_exemplo)
print("dZ1 =", dZ1_exemplo)

## Etapa 13 — Implementar o Backpropagation

O Backpropagation aplica a regra da cadeia para calcular como cada peso e bias influenciou a perda.

Para sigmoid na saída combinada com BCE média, usamos:

\[
dZ^{[2]}=(A^{[2]}-Y)/m.
\]

Depois, calculamos gradientes da camada de saída, propagamos o gradiente para a camada oculta e aplicamos a derivada da sigmoid: `A1 * (1 - A1)`. O produto `*` nesta etapa é elemento a elemento.

In [ ]:
def backward(Y, params, cache):
    m = Y.shape[0]

    X = cache["X"]
    A1 = cache["A1"]
    A2 = cache["A2"]

    dZ2 = (A2 - Y) / m

    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * A1 * (1 - A1)

    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return {
        "W1": dW1,
        "b1": db1,
        "W2": dW2,
        "b2": db2
    }

gradientes_iniciais = backward(Y, params, cache_inicial)
for nome, gradiente in gradientes_iniciais.items():
    print(nome, "gradiente:", gradiente.shape, "parâmetro:", params[nome].shape)

## Etapa 14 — Implementar a atualização dos parâmetros

O Gradient Descent atualiza cada parâmetro na direção oposta ao gradiente:

\[
\theta\leftarrow\theta-\eta\nabla_\theta L.
\]

A função abaixo percorre os quatro parâmetros. Importante: primeiro calculamos **todos** os gradientes com os parâmetros atuais; somente depois aplicamos as atualizações.

In [ ]:
def update_parameters(params, gradients, learning_rate):
    for key in params:
        params[key] -= learning_rate * gradients[key]

    return params

## Etapa 15 — Definir os parâmetros de treinamento

Usaremos os mesmos valores da apresentação:
- Taxa de aprendizado: `0.5`.
- Épocas: `20000`.
- Treinamento em lote completo: as quatro amostras passam juntas pelo forward e pelo backward em cada época.
- Histórico: armazenaremos a perda a cada 100 épocas.

In [ ]:
learning_rate = 0.5
epochs = 20000
historico_loss = []

## Etapa 16 — Treinar a MLP

Em cada época:
1. Forward para obter `A2` e o cache.
2. Calcular a perda.
3. Calcular os gradientes.
4. Atualizar pesos e biases.
5. Registrar a perda periodicamente.

A perda registrada é a perda calculada antes da atualização daquela época. Ao final, recalcularemos a perda com os parâmetros finais.

In [ ]:
for epoca in range(epochs):
    A2, cache = forward(X, params)
    loss = calcular_loss(Y, A2)
    gradients = backward(Y, params, cache)
    params = update_parameters(params, gradients, learning_rate)

    if epoca % 100 == 0:
        historico_loss.append(loss)

probabilidades_finais, _ = forward(X, params)
loss_final = calcular_loss(Y, probabilidades_finais)

print(f"Perda inicial: {loss_inicial:.6f}")
print(f"Perda final:   {loss_final:.6f}")

## Etapa 17 — Visualizar a curva de perda

O gráfico deve mostrar a Binary Cross-Entropy ao longo das épocas. Uma curva decrescente indica que a rede está ajustando seus parâmetros para reduzir o erro nas quatro combinações do XOR.

In [ ]:
epocas_registradas = np.arange(len(historico_loss)) * 100

plt.figure(figsize=(8, 4.5))
plt.plot(epocas_registradas, historico_loss, color="#62CDB5", linewidth=2)
plt.xlabel("Época")
plt.ylabel("Binary Cross-Entropy")
plt.title("Aprendizado da MLP para o XOR")
plt.grid(alpha=0.25)
plt.show()

## Etapa 18 — Gerar e conferir as previsões finais

A saída da rede é uma probabilidade estimada. Para convertê-la em classe, usamos o limiar `0.5`:
- saída `>= 0.5` → classe 1;
- saída `< 0.5` → classe 0.

Compare a previsão binária com a tabela XOR original. O resultado esperado é `[0, 1, 1, 0]`.

In [ ]:
probabilidades_finais, _ = forward(X, params)
previsoes_binarias = (probabilidades_finais >= 0.5).astype(int)

print("Entrada | Probabilidade | Previsto | Esperado")
for entrada, prob, previsto, esperado in zip(
    X, probabilidades_finais.ravel(),
    previsoes_binarias.ravel(), Y.ravel()
):
    print(f"{entrada.astype(int)} | {prob:.6f} | {previsto} | {int(esperado)}")

print("\nPrevisões binárias:", previsoes_binarias.ravel())
print("Rótulos esperados: ", Y.astype(int).ravel())
print("Acertos:", np.sum(previsoes_binarias == Y), "de", len(Y))

## Etapa 19 — Visualizar a região de decisão da rede treinada

Criamos uma grade com 300 × 300 pontos no plano de entrada. Cada ponto passa pelo mesmo `forward`; por isso, a fronteira representa o modelo que foi efetivamente treinado.

- `meshgrid` monta as coordenadas da grade.
- `column_stack` transforma as coordenadas em 90.000 linhas com duas entradas.
- `forward` calcula a saída da rede para todos os pontos.
- `reshape` devolve as saídas ao formato 300 × 300.
- A linha de nível `0.5` separa as regiões previstas como classe 0 e classe 1.

In [ ]:
x1 = np.linspace(-0.5, 1.5, 300)
x2 = np.linspace(-0.5, 1.5, 300)
G1, G2 = np.meshgrid(x1, x2)

grade = np.column_stack([G1.ravel(), G2.ravel()])
saida_grade, _ = forward(grade, params)
saida_grade = saida_grade.reshape(G1.shape)

plt.figure(figsize=(7, 5.5))
plt.contourf(G1, G2, saida_grade, levels=20, cmap="coolwarm", alpha=0.85)
plt.contour(G1, G2, saida_grade, levels=[0.5], colors="black", linewidths=2)
plt.scatter(X[:, 0], X[:, 1], c=Y.ravel(), cmap="coolwarm",
            s=220, edgecolors="black", linewidths=1.2)
for (x1_pt, x2_pt), y in zip(X, Y.ravel()):
    plt.annotate(str(int(y)), (x1_pt, x2_pt), xytext=(0, 9),
                 textcoords="offset points", ha="center", fontsize=11)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Região de decisão da MLP treinada para XOR")
plt.grid(alpha=0.15)
plt.show()

## Etapa 20 — Conclusão e verificações

Neste exercício, observamos duas capacidades diferentes:

| Perceptron | MLP |
|---|---|
| Um único neurônio | Camada oculta e camada de saída |
| Aprende fronteira linear | Pode aprender fronteiras não lineares |
| Aprende OR | Aprende XOR com a arquitetura e o treinamento usados aqui |
| Regra de atualização do Perceptron | Backpropagation + Gradient Descent |

O que torna a MLP mais expressiva é a composição de transformações lineares com ativações não lineares na camada oculta.

Vamos executar uma verificação final. Se o treinamento convergiu como esperado, a asserção será aprovada.

In [ ]:
resultado_esperado = np.array([0, 1, 1, 0])
resultado_obtido = previsoes_binarias.ravel()

assert np.array_equal(resultado_obtido, resultado_esperado), (
    f"A MLP não reproduziu XOR. Obtido: {resultado_obtido}; esperado: {resultado_esperado}"
)
assert loss_final < loss_inicial, "A perda final deveria ser menor que a inicial."

print("Verificação aprovada: a MLP aprendeu as quatro combinações do XOR.")
print(f"Perda: {loss_inicial:.6f} → {loss_final:.6f}")

---

### Observações para o professor

- Rode as células na ordem, começando pelos imports.
- O Perceptron OR usa rótulos `-1/+1`; a MLP XOR usa rótulos `0/1` porque a saída sigmoid e a Binary Cross-Entropy foram formuladas para esse intervalo.
- Não compare diretamente a saída `-1/+1` do Perceptron com a saída probabilística `0..1` da MLP sem converter a representação.
- O resultado numérico do treinamento depende dos parâmetros e da implementação. Este notebook fixa a semente e os hiperparâmetros da aula para tornar a execução reproduzível.
- A MLP aprende as quatro linhas do XOR neste experimento. Isso demonstra capacidade de representação e ajuste aos dados apresentados; não constitui uma avaliação de generalização para dados externos.